In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
books = pd.read_csv("../data/processed/books_enhanced.csv")

tag_matrix = pd.read_pickle("../data/processed/tag_matrix_tfidf.pkl")

content_features = pd.read_pickle("../data/processed/content_features.pkl")

In [4]:
content_features = content_features.set_index("book_id")

books = books.set_index("book_id")

In [5]:
tfidf_text = TfidfVectorizer(stop_words="english")

content_tfidf_matrix = tfidf_text.fit_transform(
    content_features["content_features"]
)

In [6]:
tag_similarity = cosine_similarity(tag_matrix)

In [7]:
content_similarity = cosine_similarity(content_tfidf_matrix)

In [8]:
tag_similarity_df = pd.DataFrame(
    tag_similarity,
    index=tag_matrix.index,
    columns=tag_matrix.index
)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=content_features.index,
    columns=content_features.index
)

In [9]:
tag_density = tag_matrix.sum(axis=1).mean()
content_density = content_tfidf_matrix.sum(axis=1).mean()

total = tag_density + content_density

tag_weight = tag_density / total
content_weight = content_density / total

print("Tag Weight:", round(tag_weight,3))
print("Content Weight:", round(content_weight,3))

Tag Weight: 0.766
Content Weight: 0.234


In [10]:
hybrid_similarity = (
    tag_weight * tag_similarity_df +
    content_weight * content_similarity_df
)

In [11]:
def recommend_books_content(book_id, top_n=10):
    
    if book_id not in hybrid_similarity.columns:
        return "Book not found"

    scores = hybrid_similarity[book_id].sort_values(ascending=False)

    recommended_ids = scores.iloc[1:top_n+1].index

    recommendations = books.loc[recommended_ids][
        ["title","authors","average_rating","popularity_score"]
    ]

    return recommendations

In [12]:

def precision_at_k(book_id, k=10):
    
    recommended_ids = hybrid_similarity[book_id].sort_values(
        ascending=False
    ).iloc[1:k+1].index
    
    # Relevant books = top similar books (threshold based)
    relevant = hybrid_similarity[book_id] > 0.5
    
    relevant_ids = set(hybrid_similarity[book_id][relevant].index)
    
    recommended_set = set(recommended_ids)
    
    precision = len(recommended_set & relevant_ids) / k
    
    return round(precision,3)

In [13]:
precision_at_k(1,10)

0.7

In [14]:
precision_at_k(10,10)


0.7

In [15]:
#isse kl dekhna h
precision_at_k(50,10)


1.0

In [16]:

def mmr_rerank(book_id, top_n=10, lambda_param=0.7):
    
    candidates = hybrid_similarity[book_id].sort_values(
        ascending=False
    ).iloc[1:].index.tolist()
    
    selected = []
    
    while len(selected) < top_n and candidates:
        
        if not selected:
            selected.append(candidates.pop(0))
        else:
            
            mmr_scores = []
            
            for candidate in candidates:
                
                relevance = hybrid_similarity.loc[book_id, candidate]
                
                diversity = max(
                    hybrid_similarity.loc[candidate, s]
                    for s in selected
                )
                
                score = lambda_param * relevance - \
                        (1 - lambda_param) * diversity
                
                mmr_scores.append((candidate, score))
            
            best_candidate = max(mmr_scores, key=lambda x: x[1])[0]
            
            selected.append(best_candidate)
            candidates.remove(best_candidate)
    
    return books.loc[selected][["title","authors"]]

In [18]:
mmr_rerank(1)

,title,authors
book_id,,
3,"Twilight (Twilight, #1)",Stephenie Meyer
6,The Fault in Our Stars,John Green
5,The Great Gatsby,F. Scott Fitzgerald
2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré"
10,Pride and Prejudice,Jane Austen
2002,The Death of Ivan Ilych,"Leo Tolstoy, Aylmer Maude"
8,The Catcher in the Rye,J.D. Salinger
8252,"Feral Sins (The Phoenix Pack, #1)",Suzanne Wright
9475,Adam,Ted Dekker


In [19]:
def cold_start_recommendation(top_n=10):
    
    return books.sort_values(
        ["average_rating","popularity_score"],
        ascending=False
    ).head(top_n)[["title","authors"]]

In [20]:
recommend_books_content(1)

,title,authors,average_rating,popularity_score
goodreads_book_id,,,,
3,"Twilight (Twilight, #1)",Stephenie Meyer,3.57,3928.0
5,The Great Gatsby,F. Scott Fitzgerald,3.89,3975.0
6,The Fault in Our Stars,John Green,4.26,5.0
2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",4.44,4923.0
10,Pride and Prejudice,Jane Austen,4.24,3466.0
2002,The Death of Ivan Ilych,"Leo Tolstoy, Aylmer Maude",4.04,127.0
8,The Catcher in the Rye,J.D. Salinger,3.79,3923.0
6294,Private Games (Private #3),"James Patterson, Mark T. Sullivan",3.83,0.0
9475,Adam,Ted Dekker,3.93,29.0


In [44]:
print(type(hybrid_similarity))
print(hybrid_similarity.shape)

<class 'pandas.DataFrame'>
(19188, 19188)


In [48]:
import os
import pickle

# Create models folder if not exists
os.makedirs("models", exist_ok=True)

with open("models/similarity_matrix.pkl", "wb") as f:
    pickle.dump(hybrid_similarity, f)

print("Similarity matrix saved successfully!")

Similarity matrix saved successfully!
